In [ ]:
! pip install opencv-python pillow

In [ ]:

import tkinter as tk
from tkinter import filedialog, messagebox
import cv2
from PIL import Image, ImageTk
import numpy as np


# ==========================================
# QR CODE IMAGE READER APPLICATION
# ==========================================

class QRCodeImageReader:

    def __init__(self, root):

        self.root = root
        self.root.title("QR Code Image Reader")
        self.root.geometry("1000x800")

        self.image = None
        self.photo = None
        self.image_path = None

        # Create QR Code Detector
        self.qr_detector = cv2.QRCodeDetector()

        # ==================================
        # TITLE
        # ==================================

        title = tk.Label(
            root,
            text="QR Code Image Reader",
            font=("Arial", 22, "bold")
        )
        title.pack(pady=15)

        # ==================================
        # UPLOAD IMAGE BUTTON
        # ==================================

        upload_button = tk.Button(
            root,
            text="Upload QR Code Image",
            font=("Arial", 13, "bold"),
            width=25,
            command=self.upload_image
        )

        upload_button.pack(pady=10)

        # ==================================
        # IMAGE DISPLAY
        # ==================================

        self.image_label = tk.Label(
            root,
            text="Upload an image containing a QR code",
            bg="black",
            fg="white",
            width=80,
            height=25
        )

        self.image_label.pack(pady=15)

        # ==================================
        # READ QR CODE BUTTON
        # ==================================

        read_button = tk.Button(
            root,
            text="Read QR Code",
            font=("Arial", 13, "bold"),
            width=25,
            command=self.read_qr
        )

        read_button.pack(pady=10)

        # ==================================
        # RESULT LABEL
        # ==================================

        result_title = tk.Label(
            root,
            text="Decoded QR Code Information:",
            font=("Arial", 13, "bold")
        )

        result_title.pack(pady=5)

        # ==================================
        # RESULT TEXT BOX
        # ==================================

        self.result_text = tk.Text(
            root,
            height=5,
            width=90,
            font=("Arial", 12)
        )

        self.result_text.pack(pady=10)

        # ==================================
        # STATUS
        # ==================================

        self.status_label = tk.Label(
            root,
            text="Waiting for image...",
            font=("Arial", 11)
        )

        self.status_label.pack(pady=10)

    # ==========================================
    # UPLOAD IMAGE
    # ==========================================

    def upload_image(self):

        file_path = filedialog.askopenfilename(

            title="Select an Image Containing a QR Code",

            filetypes=[
                ("Image Files", "*.png *.jpg *.jpeg *.bmp"),
                ("All Files", "*.*")
            ]
        )

        if not file_path:
            return

        # Read image using OpenCV
        image = cv2.imread(file_path)

        if image is None:

            messagebox.showerror(
                "Error",
                "Unable to open the selected image."
            )

            return

        self.image = image
        self.image_path = file_path

        # Convert BGR to RGB
        image_rgb = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

        # Convert to PIL image
        pil_image = Image.fromarray(image_rgb)

        # Resize for GUI preview while preserving proportions
        pil_image.thumbnail((850, 450))

        self.photo = ImageTk.PhotoImage(pil_image)

        # Display uploaded image
        self.image_label.config(
            image=self.photo,
            text=""
        )

        # Clear previous result
        self.result_text.delete("1.0", tk.END)

        self.status_label.config(
            text="Image uploaded successfully. Click Read QR Code."
        )

    # ==========================================
    # READ QR CODE
    # ==========================================

    def read_qr(self):

        if self.image is None:

            messagebox.showwarning(
                "No Image",
                "Please upload an image first."
            )

            return

        # Convert image to grayscale
        gray = cv2.cvtColor(
            self.image,
            cv2.COLOR_BGR2GRAY
        )

        results = []

        # ----------------------------------
        # TRY MULTIPLE QR CODE DETECTION
        # ----------------------------------

        try:

            found, decoded_info, points, _ = (
                self.qr_detector.detectAndDecodeMulti(gray)
            )

            if found:

                for i, text in enumerate(decoded_info):

                    if text:

                        results.append(text)

                        # Draw green boundary
                        polygon = points[i].astype(int)

                        cv2.polylines(
                            self.image,
                            [polygon],
                            True,
                            (0, 255, 0),
                            4
                        )

        except cv2.error:

            pass

        # ----------------------------------
        # TRY SINGLE QR CODE DETECTION
        # ----------------------------------

        if not results:

            text, points, _ = (
                self.qr_detector.detectAndDecode(gray)
            )

            if text:

                results.append(text)

                if points is not None:

                    polygon = points.astype(int)

                    cv2.polylines(
                        self.image,
                        [polygon],
                        True,
                        (0, 255, 0),
                        4
                    )

        # ----------------------------------
        # DISPLAY RESULTS
        # ----------------------------------

        self.result_text.delete(
            "1.0",
            tk.END
        )

        if results:

            for i, text in enumerate(results):

                self.result_text.insert(
                    tk.END,
                    f"QR Code {i + 1}:\n{text}\n\n"
                )

            self.status_label.config(
                text=f"Successfully decoded {len(results)} QR code(s)!"
            )

            # Refresh image to display green boundary
            self.display_image()

        else:

            self.status_label.config(
                text="No QR code detected in this image."
            )

            messagebox.showwarning(
                "QR Code Not Found",
                "Could not detect a readable QR code.\n"
                "Try a clearer or higher-resolution image."
            )

    # ==========================================
    # DISPLAY IMAGE
    # ==========================================

    def display_image(self):

        image_rgb = cv2.cvtColor(
            self.image,
            cv2.COLOR_BGR2RGB
        )

        pil_image = Image.fromarray(image_rgb)

        pil_image.thumbnail((850, 450))

        self.photo = ImageTk.PhotoImage(pil_image)

        self.image_label.config(
            image=self.photo
        )


# ==========================================
# MAIN PROGRAM
# ==========================================

if __name__ == "__main__":

    root = tk.Tk()

    app = QRCodeImageReader(root)

    root.mainloop()
